# CP_19 — archived capstone notebook

Run from the repository root. Stored outputs are historical evidence, not a fresh execution. CP_20 and CP_21 setup paths were made portable; select your own project kernel. Earlier notebooks retain their original code. See docs/REPRODUCIBILITY.md.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

project_path = Path.cwd()
initial_data_path = project_path / "initial_data"
feedback_path = project_path / "week_7"

inputs_path = feedback_path / "inputs.txt"
outputs_path = feedback_path / "outputs.txt"

completed_rounds = 7
submission_round = 8

print("Python executable:", sys.executable)
print("Working directory:", project_path)
print("Initial data folder found:", initial_data_path.is_dir())
print("Cumulative inputs found:", inputs_path.is_file())
print("Cumulative outputs found:", outputs_path.is_file())
print("Completed rounds:", completed_rounds)
print("Round to prepare:", submission_round)

assert initial_data_path.is_dir(), "Initial data folder not found."
assert inputs_path.is_file(), "week_7/inputs.txt not found."
assert outputs_path.is_file(), "week_7/outputs.txt not found."

Python executable: c:\Users\AG\BeatrizGIT\Master\.venv\Scripts\python.exe
Working directory: c:\Users\AG\BeatrizGIT\Master\CapstoneProject
Initial data folder found: True
Cumulative inputs found: True
Cumulative outputs found: True
Completed rounds: 7
Round to prepare: 8


This cell identifies the Python interpreter and working directory, imports
the libraries needed for data handling, and checks that the initial data
folder and cumulative feedback files are available.

The notebook is configured to use seven completed evaluation rounds to
prepare the eighth submission. The contents of the files will be checked
in the next step before constructing the updated datasets.

*Activating the virtual environment in the terminal and selecting the
notebook kernel are separate steps. The printed executable confirms
which Python environment is actually running this notebook.*

In [2]:
import ast
import re


def read_feedback_rounds(path):
    """Read cumulative rounds without executing the file contents."""
    text = path.read_text(encoding="utf-8-sig").strip()

    # Convert the array([...]) notation into ordinary Python lists.
    text = re.sub(r"array\((\[[^\[\]]*\])\)", r"\1", text)

    tree = ast.parse(text, mode="exec")
    rounds = []

    for node in tree.body:
        if not isinstance(node, ast.Expr):
            raise ValueError(f"Unexpected file structure: {path}")

        rounds.append(ast.literal_eval(node.value))

    return rounds


input_rounds = read_feedback_rounds(inputs_path)
output_rounds = read_feedback_rounds(outputs_path)

assert len(input_rounds) == completed_rounds, (
    f"Expected {completed_rounds} input rounds, found {len(input_rounds)}."
)
assert len(output_rounds) == completed_rounds, (
    f"Expected {completed_rounds} output rounds, found {len(output_rounds)}."
)
assert all(len(r) == 8 for r in input_rounds), (
    "Each input round must contain eight functions."
)
assert all(len(r) == 8 for r in output_rounds), (
    "Each output round must contain eight functions."
)

expected_dimensions = {
    1: 2, 2: 2, 3: 3, 4: 4,
    5: 4, 6: 5, 7: 6, 8: 8
}

# Rebuild the datasets on every execution to avoid appending rounds twice.
initial_data = {}
updated_data = {}

for function in range(1, 9):
    folder = initial_data_path / f"function_{function}"
    dimension = expected_dimensions[function]

    X_initial = np.load(
        folder / "initial_inputs.npy", allow_pickle=False
    )
    y_initial = np.load(
        folder / "initial_outputs.npy", allow_pickle=False
    ).reshape(-1)

    assert X_initial.ndim == 2
    assert X_initial.shape == (len(y_initial), dimension)

    X_new = np.asarray(
        [r[function - 1] for r in input_rounds],
        dtype=float
    )
    y_new = np.asarray(
        [r[function - 1] for r in output_rounds],
        dtype=float
    )

    assert X_new.shape == (completed_rounds, dimension)
    assert y_new.shape == (completed_rounds,)

    X = np.vstack([X_initial, X_new])
    y = np.concatenate([y_initial, y_new])

    assert np.isfinite(X).all(), (
        f"Non-finite inputs in Function {function}."
    )
    assert np.isfinite(y).all(), (
        f"Non-finite outputs in Function {function}."
    )
    assert ((X_new >= 0) & (X_new < 1)).all(), (
        f"Submitted inputs outside the permitted range: Function {function}."
    )

    initial_data[function] = {
        "X": X_initial,
        "y": y_initial
    }
    updated_data[function] = {
        "X": X,
        "y": y
    }

    print(
        f"Function {function}: "
        f"{len(y_initial)} → {len(y)} observations "
        f"| {dimension} variables"
    )

Function 1: 10 → 17 observations | 2 variables
Function 2: 10 → 17 observations | 2 variables
Function 3: 15 → 22 observations | 3 variables
Function 4: 30 → 37 observations | 4 variables
Function 5: 20 → 27 observations | 4 variables
Function 6: 20 → 27 observations | 5 variables
Function 7: 30 → 37 observations | 6 variables
Function 8: 40 → 47 observations | 8 variables


This cell loads the initial observations and the seven completed feedback
rounds. It verifies that each round contains eight functions, checks input
dimensions and numerical validity, and confirms that submitted coordinates
are within the permitted range.

The updated datasets contain 17, 17, 22, 37, 27, 27, 37 and 47 observations
for Functions 1–8, respectively. These totals differ because the original
datasets contained different numbers of observations. All outputs,
including negative and very small values, are retained.

*The datasets are rebuilt from the initial files and a single cumulative
feedback snapshot on every execution. This prevents repeated execution
from appending the same rounds twice. Scientific notation in the feedback
files is parsed numerically rather than copied manually.*

In [3]:
progress_rows = []

for function in range(1, 9):
    best_so_far = float(initial_data[function]["y"].max())

    for round_index in range(completed_rounds):
        observed_output = float(output_rounds[round_index][function - 1])
        improved = observed_output > best_so_far
        current_best = max(best_so_far, observed_output)

        progress_rows.append({
            "Function": function,
            "Round": round_index + 1,
            "Best before round": best_so_far,
            "Observed output": observed_output,
            "Improved": improved,
            "Improvement": current_best - best_so_far,
            "Current best": current_best
        })

        best_so_far = current_best

    assert best_so_far == float(updated_data[function]["y"].max())

progress_summary = pd.DataFrame(progress_rows)

round_7_summary = (
    progress_summary.loc[
        progress_summary["Round"] == completed_rounds
    ]
    .drop(columns="Round")
    .set_index("Function")
    .rename(columns={
        "Best before round": "Best before round 7",
        "Observed output": "Round 7 output",
        "Improved": "Round 7 improved"
    })
)

display(
    round_7_summary.style.format({
        "Best before round 7": "{:.6e}",
        "Round 7 output": "{:.6e}",
        "Improvement": "{:.6e}",
        "Current best": "{:.6e}"
    })
)

for round_number in range(1, completed_rounds + 1):
    improved_functions = progress_summary.loc[
        (progress_summary["Round"] == round_number)
        & progress_summary["Improved"],
        "Function"
    ].tolist()

    print(
        f"Functions improved in round {round_number}: "
        f"{improved_functions} | Count: {len(improved_functions)}"
    )

,Best before round 7,Round 7 output,Round 7 improved,Improvement,Current best
Function,,,,,
1,7.710875e-16,1.234775e-227,False,0.000000e+00,7.710875e-16
2,6.582395e-01,5.824477e-01,False,0.000000e+00,6.582395e-01
3,-3.096391e-02,-1.197518e-01,False,0.000000e+00,-3.096391e-02
4,4.098217e-01,1.219422e-01,False,0.000000e+00,4.098217e-01
5,6.584614e+03,8.343789e+03,True,1.759175e+03,8.343789e+03
6,-2.424310e-01,-5.714810e-01,False,0.000000e+00,-2.424310e-01
7,2.171300e+00,5.106856e-01,False,0.000000e+00,2.171300e+00
8,9.922955e+00,9.893226e+00,False,0.000000e+00,9.922955e+00


Functions improved in round 1: [2, 4, 5, 7, 8] | Count: 5
Functions improved in round 2: [6, 8] | Count: 2
Functions improved in round 3: [5, 8] | Count: 2
Functions improved in round 4: [6, 8] | Count: 2
Functions improved in round 5: [3, 4, 5, 8] | Count: 4
Functions improved in round 6: [5, 6, 7] | Count: 3
Functions improved in round 7: [5] | Count: 1


This cell compares each new observation with the best output available
before its evaluation. Because all functions are maximisation tasks,
an improvement is recorded only when the new output exceeds the
previous best. The cumulative best is retained when a query performs worse.

In Round 7, only Function 5 improved, increasing from approximately
6,584.61 to 8,343.79, an absolute gain of approximately 1,759.17.
The other seven functions retained their previous best observations.

This outcome provides evidence of progress for Function 5, but does not
establish that the hyperparameter changes in Module 18 improved the
overall strategy. Each function received only one new evaluation, and
there was no simultaneous evaluation of an alternative strategy.

*The number of improving functions and the magnitude of improvement
measure optimisation progress. They do not directly measure predictive
accuracy or uncertainty calibration, which will be reviewed separately.*

In [4]:
# Historical predictions from CP_18, recorded before Round 7 feedback.
# These displayed values are rounded.
round_7_predictions = {
    1: {"mean": -2.018961e-04, "std": 9.198985e-04},
    2: {"mean": 6.026122e-01, "std": 8.218702e-02},
    3: {"mean": -5.617723e-02, "std": 7.579168e-02},
    4: {"mean": 2.147802e-01, "std": 2.415637e-01},
    5: {
        "mean": 7.002326e+03,
        "std": 1.631969e+02,
        "auxiliary": 6.903573e+03
    },
    6: {"mean": -2.809582e-01, "std": 1.319670e-01},
    7: {
        "mean": 2.129202e+00,
        "std": 1.734245e-01,
        "auxiliary": 9.154697e-01
    },
    8: {"mean": 9.860398e+00, "std": 7.846440e-02}
}

round_7_expected_queries = {
    1: "0.166230-0.964592",
    2: "0.691822-0.000024",
    3: "0.996398-0.949282-0.003798",
    4: "0.401554-0.384590-0.372748-0.391438",
    5: "0.995096-0.996020-0.994275-0.997892",
    6: "0.476133-0.383212-0.776992-0.568645-0.009427",
    7: "0.048671-0.193593-0.957056-0.184874-0.553329-0.753430",
    8: (
        "0.065894-0.388254-0.042724-0.117374-"
        "0.911203-0.566412-0.165832-0.270788"
    )
}

review_rows = []

for function in range(1, 9):
    expected_query = np.array([
        float(value)
        for value in round_7_expected_queries[function].split("-")
    ])

    recorded_query = np.asarray(
        input_rounds[6][function - 1], dtype=float
    )

    # Confirm that the prediction refers to the input actually evaluated.
    np.testing.assert_allclose(
        recorded_query,
        expected_query,
        rtol=0,
        atol=1e-9,
        err_msg=f"Round 7 query mismatch for Function {function}"
    )

    prediction = round_7_predictions[function]
    observed = float(output_rounds[6][function - 1])
    mean = prediction["mean"]
    std = prediction["std"]
    auxiliary = prediction.get("auxiliary", np.nan)

    assert std > 0
    error = observed - mean

    review_rows.append({
        "Function": function,
        "Predicted mean": mean,
        "Predicted std": std,
        "Observed output": observed,
        "Observed minus predicted": error,
        "Absolute error / predicted std": abs(error) / std,
        "GP absolute error": abs(error),
        "Auxiliary prediction": auxiliary,
        "Auxiliary absolute error": (
            abs(observed - auxiliary)
            if np.isfinite(auxiliary)
            else np.nan
        )
    })

prediction_review = pd.DataFrame(review_rows).set_index("Function")

formats = {
    column: "{:.6e}" for column in prediction_review.columns
}
formats["Absolute error / predicted std"] = "{:.2f}"

display(prediction_review.style.format(formats, na_rep="—"))

,Predicted mean,Predicted std,Observed output,Observed minus predicted,Absolute error / predicted std,GP absolute error,Auxiliary prediction,Auxiliary absolute error
Function,,,,,,,,
1,-2.018961e-04,9.198985e-04,1.234775e-227,2.018961e-04,0.22,2.018961e-04,—,—
2,6.026122e-01,8.218702e-02,5.824477e-01,-2.016451e-02,0.25,2.016451e-02,—,—
3,-5.617723e-02,7.579168e-02,-1.197518e-01,-6.357454e-02,0.84,6.357454e-02,—,—
4,2.147802e-01,2.415637e-01,1.219422e-01,-9.283805e-02,0.38,9.283805e-02,—,—
5,7.002326e+03,1.631969e+02,8.343789e+03,1.341463e+03,8.22,1.341463e+03,6.903573e+03,1.440216e+03
6,-2.809582e-01,1.319670e-01,-5.714810e-01,-2.905228e-01,2.20,2.905228e-01,—,—
7,2.129202e+00,1.734245e-01,5.106856e-01,-1.618516e+00,9.33,1.618516e+00,9.154697e-01,4.047841e-01
8,9.860398e+00,7.846440e-02,9.893226e+00,3.282827e-02,0.42,3.282827e-02,—,—


This cell compares the predictions recorded in CP_18 with the outputs
returned for Round 7. It first verifies that each historical prediction
corresponds to the input actually evaluated.

Signed errors distinguish underestimation from overestimation. Absolute
errors are divided by the predicted standard deviation to show the size
of each discrepancy relative to the GP's reported uncertainty. Auxiliary
predictions are also assessed for Functions 5 and 7.

These checks separate optimisation success from predictive accuracy:
a query can improve the best observed output while producing a large
prediction error.

*The historical predictions are kept unchanged and use the rounded
values displayed in CP_18. No model is refitted in this cell. One
observation per function cannot establish uncertainty calibration, but
large discrepancies identify cases that warrant closer examination.*

Round 7 shows that optimisation progress and predictive reliability can
diverge. Function 5 achieved a new best output, but the GP underestimated
it by approximately 1,341.46, equivalent to 8.22 predictive standard
deviations. The auxiliary neural network also underestimated the output
and had a slightly larger absolute error.

Function 7 showed the largest standardised discrepancy. The GP predicted
approximately 2.1292, while the observed output was 0.5107, giving an
error of 9.33 predictive standard deviations. The auxiliary linear model
was closer, although it also overestimated the result. Function 6 was
overestimated by 2.20 predictive standard deviations.

For Functions 1, 2, 3, 4 and 8, the errors were below one predictive
standard deviation. These individual outcomes do not establish that
their uncertainty estimates are well calibrated. Function 2's smaller
error than in Round 6 concerns a different query and cannot be attributed
solely to hyperparameter tuning.

These results reinforce the need to distinguish observed improvement,
prediction accuracy and uncertainty. Agreement between the GP and neural
network for Function 5 did not ensure an accurate estimate of the
improvement's magnitude.

*For the Module 19 prompting exercise, these numerical results will serve
as evidence against which LLM explanations can be checked. Explanations
must distinguish predictions from observations and must not describe
model agreement or a high UCB score as a guarantee of success.*

In [5]:
import warnings
from time import perf_counter

from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel, Matern, WhiteKernel
)

# Retain the configurations selected in CP_18.
round_8_gp_configs = {
    1: {"name": "Matern_1.5_original", "nu": 1.5, "bounds": (0.02, 2.0)},
    2: {"name": "Matern_1.5_wider", "nu": 1.5, "bounds": (0.02, 10.0)},
    3: {"name": "Matern_1.5_original", "nu": 1.5, "bounds": (0.02, 2.0)},
    4: {"name": "Matern_1.5_wider", "nu": 1.5, "bounds": (0.02, 10.0)},
    5: {"name": "Matern_1.5_original", "nu": 1.5, "bounds": (0.02, 2.0)},
    6: {"name": "Matern_2.5_original", "nu": 2.5, "bounds": (0.02, 2.0)},
    7: {"name": "Matern_2.5_original", "nu": 2.5, "bounds": (0.02, 2.0)},
    8: {"name": "Matern_2.5_wider", "nu": 2.5, "bounds": (0.02, 10.0)}
}

round_8_gp_models = {}
fit_rows = []
warning_rows = []

for function in range(1, 9):
    X = updated_data[function]["X"]
    y = updated_data[function]["y"]
    config = round_8_gp_configs[function]

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X.shape[1],
            length_scale_bounds=config["bounds"],
            nu=config["nu"]
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=3,
        random_state=42
    )

    start = perf_counter()

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X, y)

    fit_seconds = perf_counter() - start
    round_8_gp_models[function] = model

    for warning in caught:
        warning_rows.append({
            "Function": function,
            "Category": warning.category.__name__,
            "Message": str(warning.message)
        })

    fit_rows.append({
        "Function": function,
        "Observations": len(y),
        "GP configuration": config["name"],
        "Fitted kernel": str(model.kernel_),
        "Log-marginal likelihood": model.log_marginal_likelihood_value_,
        "GP fit seconds": fit_seconds
    })

round_8_fit_summary = pd.DataFrame(fit_rows).set_index("Function")
round_8_fit_warnings = pd.DataFrame(
    warning_rows,
    columns=["Function", "Category", "Message"]
)

display(
    round_8_fit_summary.style.format({
        "Log-marginal likelihood": "{:.6f}",
        "GP fit seconds": "{:.3f}"
    })
)

# Print complete warnings to avoid truncated table messages.
if round_8_fit_warnings.empty:
    print("No fitting warnings.")
else:
    print("\nFull fitting warnings:")
    for row in round_8_fit_warnings.drop_duplicates().itertuples(index=False):
        print(f"\nFunction {row.Function} — {row.Category}")
        print(row.Message)

,Observations,GP configuration,Fitted kernel,Log-marginal likelihood,GP fit seconds
Function,,,,,
1,17,Matern_1.5_original,"1.06**2 * Matern(length_scale=[0.02, 2], nu=1.5) + WhiteKernel(noise_level=5.23e-08)",-22.802907,0.363
2,17,Matern_1.5_wider,"0.998**2 * Matern(length_scale=[0.0945, 10], nu=1.5) + WhiteKernel(noise_level=0.0575)",-15.731707,0.098
3,22,Matern_1.5_original,"1.16**2 * Matern(length_scale=[2, 2, 0.0725], nu=1.5) + WhiteKernel(noise_level=1e-06)",-26.000075,0.148
4,37,Matern_1.5_wider,"4.98**2 * Matern(length_scale=[4.01, 3.67, 3.69, 3.83], nu=1.5) + WhiteKernel(noise_level=1.38e-05)",-1.035432,0.229
5,27,Matern_1.5_original,"1.82**2 * Matern(length_scale=[1.92, 2, 1.71, 2], nu=1.5) + WhiteKernel(noise_level=0.0191)",-14.731415,0.131
6,27,Matern_2.5_original,"1.73**2 * Matern(length_scale=[1, 1.32, 1.38, 1.2, 2], nu=2.5) + WhiteKernel(noise_level=0.0366)",-23.157053,0.165
7,37,Matern_2.5_original,"0.702**2 * Matern(length_scale=[1.17, 2, 2, 0.194, 0.131, 0.119], nu=2.5) + WhiteKernel(noise_level=6.6e-10)",-33.520608,0.236
8,47,Matern_2.5_wider,"2.95**2 * Matern(length_scale=[3.03, 4.58, 2.42, 5.19, 10, 4.91, 3.12, 10], nu=2.5) + WhiteKernel(noise_level=1e-10)",9.547558,0.524



Full fitting warnings:

Function 1 — ConvergenceWarning
The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 0.02. Decreasing the bound and calling fit again may find a better value.

Function 1 — ConvergenceWarning
The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.

Function 2 — ConvergenceWarning
The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.

Function 3 — ConvergenceWarning
The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.

Function 3 — ConvergenceWarning
The optimal value found for dimension 1 of parameter k1__k2__length_s

This cell refits the eight Gaussian Process models using the initial
observations and all seven completed evaluation rounds. It retains the
Matérn smoothness settings and length-scale bounds selected in CP_18.

Although these configuration choices remain fixed, the individual length
scales, signal amplitude and noise level are re-estimated on the updated
data. Output normalisation is performed during fitting, and three
optimiser restarts are used with a fixed random seed.

The fitted models are stored in round_8_gp_models for candidate evaluation.
The table reports the observation counts, fitted kernels, marginal
likelihoods and fitting times. Full warning messages are retained and
printed for inspection.

*This is a data update, not a new hyperparameter search or validation
experiment. Retaining the previous configurations provides continuity,
but does not establish that they remain optimal. Refitting alone also
does not demonstrate that the large prediction discrepancies observed
for Functions 5 and 7 have been resolved.*


The updated fits show different responses to the latest observations.
For Function 5, the fitted white-noise variance increased from an almost
zero value to 0.0191 in normalised output units. This may reflect variation
that the current kernel does not explain adequately; it does not establish
that the black-box evaluations are stochastic.

Function 7's length scales changed substantially again, reinforcing the
need to treat its apparent input sensitivities as provisional. Functions
1 and 8 retained broadly similar fitted patterns, including parameters
at their permitted bounds.

*These are changes in fitted model parameters, not new validation results.
Neither a changed noise estimate nor a stable kernel demonstrates that
predictive uncertainty is calibrated. The latest observations remain
evidence for evaluating subsequent predictions.*

In [7]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression

auxiliary_estimators = {
    5: (
        "NN (8,)",
        MLPRegressor(
            hidden_layer_sizes=(8,),
            activation="tanh",
            solver="lbfgs",
            alpha=0.1,
            max_iter=2000,
            random_state=42
        )
    ),
    7: ("Linear regression", LinearRegression())
}

round_8_auxiliary_models = {}
round_8_auxiliary_names = {}
auxiliary_fit_rows = []
auxiliary_warning_rows = []

for function, (name, estimator) in auxiliary_estimators.items():
    X = updated_data[function]["X"]
    y = updated_data[function]["y"]

    model = TransformedTargetRegressor(
        regressor=Pipeline([
            ("input_scaler", StandardScaler()),
            ("model", estimator)
        ]),
        transformer=StandardScaler()
    )

    start = perf_counter()

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X, y)

    fit_seconds = perf_counter() - start

    round_8_auxiliary_models[function] = model
    round_8_auxiliary_names[function] = name

    for warning in caught:
        auxiliary_warning_rows.append({
            "Function": function,
            "Category": warning.category.__name__,
            "Message": str(warning.message)
        })

    auxiliary_fit_rows.append({
        "Function": function,
        "Observations": len(y),
        "Auxiliary model": name,
        "Fit seconds": fit_seconds
    })

round_8_auxiliary_summary = (
    pd.DataFrame(auxiliary_fit_rows).set_index("Function")
)
round_8_auxiliary_warnings = pd.DataFrame(
    auxiliary_warning_rows,
    columns=["Function", "Category", "Message"]
)

display(
    round_8_auxiliary_summary.style.format({
        "Fit seconds": "{:.3f}"
    })
)

if round_8_auxiliary_warnings.empty:
    print("No auxiliary fitting warnings.")
else:
    for row in round_8_auxiliary_warnings.itertuples(index=False):
        print(f"\nFunction {row.Function} — {row.Category}")
        print(row.Message)

,Observations,Auxiliary model,Fit seconds
Function,,,
5,27,"NN (8,)",0.076
7,37,Linear regression,0.011


No auxiliary fitting warnings.


This cell updates the auxiliary neural network for Function 5 and the
auxiliary linear regression model for Function 7 using all available
observations. Their configurations remain unchanged from CP_18.

Inputs and targets are standardised during fitting, and predictions are
returned in the original output units. The models are stored for
comparison with GP predictions at proposed candidate points.

In Round 7, the GP had a slightly smaller absolute error than the neural
network for Function 5, while linear regression was closer than the GP
for Function 7. These outcomes motivate monitoring model disagreement,
but do not establish a consistently superior model.

*The auxiliary models provide diagnostic predictions. They do not supply
a calibrated uncertainty estimate and will not be added to the GP-UCB
score. This cell performs refitting, not a new validation experiment.*

In [8]:
from scipy.spatial import cKDTree

beta_round_8 = 2.0

round_8_candidate_proposals = {}
candidate_rows = []
search_time_rows = []


def predict_in_batches(model, candidates, batch_size=5000):
    """Predict in batches to limit memory usage."""
    means = []
    standard_deviations = []

    for start in range(0, len(candidates), batch_size):
        batch = candidates[start:start + batch_size]
        mean, std = model.predict(batch, return_std=True)
        means.append(mean)
        standard_deviations.append(std)

    return np.concatenate(means), np.concatenate(standard_deviations)


for function in range(1, 9):
    start = perf_counter()

    X = updated_data[function]["X"]
    y = updated_data[function]["y"]
    model = round_8_gp_models[function]

    dimension = X.shape[1]
    best_input = X[np.argmax(y)]
    current_best = float(y.max())

    rng = np.random.default_rng(42 + function)
    observation_tree = cKDTree(X)

    candidate_pools = {
        "Global": rng.uniform(
            0.0, 0.999999, size=(100000, dimension)
        )
    }

    for radius in [0.10, 0.03]:
        lower = np.maximum(0.0, best_input - radius)
        upper = np.minimum(0.999999, best_input + radius)

        candidate_pools[f"Local {radius:.2f}"] = rng.uniform(
            lower, upper, size=(10000, dimension)
        )

    round_8_candidate_proposals[function] = {}

    for search_name, candidates in candidate_pools.items():
        # Predict at the exact six-decimal submission coordinates.
        candidates = np.unique(np.round(candidates, 6), axis=0)

        nearest_distance, _ = observation_tree.query(candidates)
        keep = nearest_distance > 1e-6

        candidates = candidates[keep]
        nearest_distance = nearest_distance[keep]

        if len(candidates) == 0:
            raise ValueError(
                f"No unevaluated candidates for Function {function}, "
                f"{search_name}."
            )

        mean, std = predict_in_batches(model, candidates)
        ucb = mean + beta_round_8 * std

        assert np.isfinite(mean).all()
        assert np.isfinite(std).all()
        assert np.isfinite(ucb).all()

        index = int(np.argmax(ucb))
        query = candidates[index].copy()

        auxiliary_prediction = np.nan
        if function in round_8_auxiliary_models:
            auxiliary_prediction = float(
                round_8_auxiliary_models[function]
                .predict(query.reshape(1, -1))[0]
            )

        proposal = {
            "query": query,
            "mean": float(mean[index]),
            "std": float(std[index]),
            "ucb": float(ucb[index]),
            "nearest_distance": float(nearest_distance[index]),
            "auxiliary_prediction": auxiliary_prediction
        }

        round_8_candidate_proposals[function][search_name] = proposal

        candidate_rows.append({
            "Function": function,
            "Search": search_name,
            "Query": "-".join(f"{value:.6f}" for value in query),
            "Current best": current_best,
            "GP mean": proposal["mean"],
            "GP std": proposal["std"],
            "UCB": proposal["ucb"],
            "Nearest observation distance": proposal["nearest_distance"],
            "Auxiliary prediction": auxiliary_prediction
        })

    search_time_rows.append({
        "Function": function,
        "Candidate search seconds": perf_counter() - start
    })

round_8_candidate_comparison = pd.DataFrame(candidate_rows)
round_8_search_times = (
    pd.DataFrame(search_time_rows).set_index("Function")
)

display(
    round_8_candidate_comparison.style
    .hide(axis="index")
    .format({
        "Current best": "{:.6e}",
        "GP mean": "{:.6e}",
        "GP std": "{:.6e}",
        "UCB": "{:.6e}",
        "Nearest observation distance": "{:.6f}",
        "Auxiliary prediction": "{:.6e}"
    }, na_rep="—")
)

display(
    round_8_search_times.style.format({
        "Candidate search seconds": "{:.3f}"
    })
)

Function,Search,Query,Current best,GP mean,GP std,UCB,Nearest observation distance,Auxiliary prediction
1,Global,0.208681-0.000255,7.710875e-16,-1.899534e-04,8.939013e-04,1.597849e-03,0.130289,—
1,Local 0.10,0.763810-0.828314,7.710875e-16,-1.241916e-04,8.513174e-04,1.578443e-03,0.086804,—
1,Local 0.03,0.761019-0.761965,7.710875e-16,-1.233740e-04,8.485671e-04,1.573760e-03,0.041698,—
2,Global,0.698510-0.006055,6.582395e-01,6.023868e-01,7.081838e-02,7.440235e-01,0.004159,—
2,Local 0.10,0.698286-0.000289,6.582395e-01,6.023824e-01,7.087811e-02,7.441387e-01,0.002370,—
2,Local 0.03,0.698207-0.000240,6.582395e-01,6.023600e-01,7.089054e-02,7.441410e-01,0.002458,—
3,Global,0.991779-0.022828-0.094305,-3.096391e-02,-5.384088e-02,5.364524e-02,5.344961e-02,0.381701,—
3,Local 0.10,0.990357-0.998324-0.459088,-3.096391e-02,-3.536318e-02,4.310757e-02,5.085196e-02,0.058028,—
3,Local 0.03,0.992851-0.994512-0.475617,-3.096391e-02,-3.173652e-02,3.686029e-02,4.198406e-02,0.042725,—
4,Global,0.347954-0.402500-0.373386-0.440083,4.098217e-01,-5.533030e-02,2.273769e-01,3.994235e-01,0.062098,—


,Candidate search seconds
Function,
1,0.286
2,0.312
3,0.350
4,0.488
5,0.513
6,0.535
7,0.586
8,0.888


This cell generates 100,000 global candidates and two local pools of
10,000 candidates per function. The local pools are centred on the best
observed input, with coordinate margins of 0.10 and 0.03, restricted to
the permitted domain.

Candidates are rounded to six decimal places before prediction.
Duplicates within each pool and points within 1e-6 Euclidean distance
of an existing observation are removed. Each pool's proposal maximises
the GP score: predicted mean + 2 × predictive standard deviation.

The search budgets, random seeds and exploration weight are retained
from CP_18. The updated observations and refitted models can nevertheless
change the proposed points. Auxiliary predictions for Functions 5 and 7
are reported separately and do not influence the UCB ranking.

The resulting table will provide a common numerical context for the
Module 19 prompt comparison. Using the same evidence across prompts
will help distinguish changes in explanation or instruction-following
from changes in the underlying numerical search.

*These are model predictions, not new function evaluations. The uncertainty
estimates remain model-dependent, particularly given the large historical
errors for Functions 5 and 7. “Global” and “local” identify how candidates
were generated; neither label alone determines whether a proposal mainly
exploits predicted performance or explores uncertainty.*

In [9]:
import json

# Build a compact context with complete query strings and numerical values.
prompt_records = []

for function in range(1, 9):
    for search_name, proposal in round_8_candidate_proposals[function].items():
        auxiliary = proposal["auxiliary_prediction"]

        prompt_records.append({
            "function": function,
            "search": search_name,
            "query": "-".join(
                f"{value:.6f}" for value in proposal["query"]
            ),
            "current_best": float(updated_data[function]["y"].max()),
            "gp_mean": proposal["mean"],
            "gp_std": proposal["std"],
            "ucb": proposal["ucb"],
            "nearest_observation_distance": proposal["nearest_distance"],
            "auxiliary_prediction": (
                float(auxiliary) if np.isfinite(auxiliary) else None
            )
        })

prompt_context = {
    "objective": "Maximise each of eight black-box functions.",
    "submission_round": 8,
    "selection_rule": "Select the highest supplied UCB per function.",
    "ucb_definition": "GP mean + 2 * GP predictive standard deviation",
    "candidates": prompt_records
}

context_json = json.dumps(
    prompt_context, indent=2, allow_nan=False
)

simple_prompt = (
    "Using the data below, select the highest-UCB candidate for each "
    "function and briefly explain the choices.\n\n"
    + context_json
)

structured_prompt = (
    "TASK\n"
    "Select exactly one candidate per function by maximising the supplied "
    "UCB. Use only the supplied data.\n\n"
    "RULES\n"
    "1. Return all eight functions, ordered from 1 to 8.\n"
    "2. Copy each selected query exactly; do not invent or change coordinates.\n"
    "3. Compare UCB values within each function, not across functions.\n"
    "4. Treat GP and auxiliary values as predictions, not observations.\n"
    "5. Do not claim that a high UCB guarantees improvement.\n"
    "6. Auxiliary predictions must not change the specified ranking rule.\n"
    "7. Do not shorten queries with ellipses.\n\n"
    "OUTPUT\n"
    "Return valid JSON only, with this structure:\n"
    '{"selections": [{"function": 1, "search": "...", '
    '"query": "...", "reason": "..."}]}\n'
    "The selections array must contain eight entries. "
    "Keep each reason to one short sentence.\n\n"
    "DATA\n"
    + context_json
)

# Python reference for checking the LLM responses later.
prompt_reference = {}

for function, proposals in round_8_candidate_proposals.items():
    search_name = max(
        proposals,
        key=lambda name: proposals[name]["ucb"]
    )
    proposal = proposals[search_name]

    prompt_reference[function] = {
        "search": search_name,
        "query": "-".join(
            f"{value:.6f}" for value in proposal["query"]
        ),
        "ucb": proposal["ucb"]
    }

prompt_experiment = {
    "simple_prompt": simple_prompt,
    "structured_prompt": structured_prompt,
    "responses": {},
    "decoding_settings": {
        "temperature": "Not configured or exposed in this chat",
        "top_p": "Not configured or exposed in this chat",
        "top_k": "Not configured or exposed in this chat",
        "max_tokens": "Not configured or exposed in this chat"
    }
}

print("Simple prompt characters:", len(simple_prompt))
print("Structured prompt characters:", len(structured_prompt))
print("Character counts are not token counts.")
print("\nSIMPLE PROMPT\n")
print(simple_prompt)

Simple prompt characters: 9170
Structured prompt characters: 9834
Character counts are not token counts.

SIMPLE PROMPT

Using the data below, select the highest-UCB candidate for each function and briefly explain the choices.

{
  "objective": "Maximise each of eight black-box functions.",
  "submission_round": 8,
  "selection_rule": "Select the highest supplied UCB per function.",
  "ucb_definition": "GP mean + 2 * GP predictive standard deviation",
  "candidates": [
    {
      "function": 1,
      "search": "Global",
      "query": "0.208681-0.000255",
      "current_best": 7.710875114502849e-16,
      "gp_mean": -0.00018995336575117536,
      "gp_std": 0.0008939013473391805,
      "ucb": 0.0015978493289271858,
      "nearest_observation_distance": 0.1302893122740334,
      "auxiliary_prediction": null
    },
    {
      "function": 1,
      "search": "Local 0.10",
      "query": "0.763810-0.828314",
      "current_best": 7.710875114502849e-16,
      "gp_mean": -0.00012419161823859

This cell prepares two zero-shot prompts using the same candidate data
and the same maximum-UCB selection rule. The simple prompt gives a brief
instruction, while the structured prompt adds explicit constraints and
a JSON output schema. No solved examples are supplied.

The candidate context is generated directly from Python variables,
preserving complete query strings and avoiding manual transcription.
A Python reference selection is also stored for later verification.

The comparison will examine selection accuracy, coordinate preservation,
completeness and unsupported claims. Differences may reflect both the
additional instructions and the requested output format; this is a
practical comparison rather than an isolated test of formatting alone.

*This cell prepares prompts but does not execute an LLM experiment.
Character counts are not token counts. Decoding parameters are recorded
as unavailable rather than assigned invented values. A requested short
response is an instruction, not a configured max_tokens limit.*

In [10]:
# Save complete prompts, avoiding notebook display truncation.
prompt_folder = project_path / "prompts" / "round_8"
prompt_folder.mkdir(parents=True, exist_ok=True)

simple_prompt_path = prompt_folder / "simple_prompt.txt"
structured_prompt_path = prompt_folder / "structured_prompt.txt"

simple_prompt_path.write_text(simple_prompt, encoding="utf-8")
structured_prompt_path.write_text(structured_prompt, encoding="utf-8")

# Confirm that saving preserved the complete contents.
assert simple_prompt_path.read_text(encoding="utf-8") == simple_prompt
assert structured_prompt_path.read_text(encoding="utf-8") == structured_prompt

print("Simple prompt saved:", simple_prompt_path)
print("Characters:", len(simple_prompt))
print()
print("Structured prompt saved:", structured_prompt_path)
print("Characters:", len(structured_prompt))

Simple prompt saved: c:\Users\AG\BeatrizGIT\Master\CapstoneProject\prompts\round_8\simple_prompt.txt
Characters: 9170

Structured prompt saved: c:\Users\AG\BeatrizGIT\Master\CapstoneProject\prompts\round_8\structured_prompt.txt
Characters: 9834


VS Code truncated the displayed prompt output. This was a notebook
display limitation, not evidence of an LLM context or output-token limit.

The complete simple and structured prompts were therefore exported to
UTF-8 text files. Reading them back and comparing them with the original
strings verifies that the saved contents are unchanged. The files will
be used to copy the prompts without relying on the shortened display.

*The prompts contain 9,170 and 9,834 characters respectively. These are
character counts, not token counts. No LLM truncation has been observed
at this stage because the prompts have not yet been submitted.*

### Round 8 — Comparing Simple and Structured Prompts

To examine how prompt design influences the interpretation of Bayesian optimisation results, I prepared two prompts using the same 24 candidate points, with three candidates per function.

The **simple prompt** provides the optimisation objective and asks for the highest-UCB candidate for each function, with a brief explanation.

The **structured prompt** explicitly defines the selection rules, requires all eight functions to be included, prohibits modifying coordinates, distinguishes predictions from observations, and requests a valid JSON response.

Both prompts use identical candidate data and the same UCB selection criterion. This allows differences in their responses to be examined in relation to instruction structure rather than differences in the underlying numerical information.

The Python-calculated maximum UCB for each function serves as the reference against which the responses will be checked. The comparison will focus on selection accuracy, coordinate preservation, output consistency and unsupported claims.

This is a comparison of prompting approaches, not a controlled experiment on decoding hyperparameters.

### Round 8 — Reviewing the Returned Responses

I reviewed the two supplied responses against the candidate comparison table and the Python reference selections.

| Check | Simple prompt response | Structured prompt response |
|:--|:--|:--|
| Correct search selection | 8/8 | 8/8 in the textual summary |
| Complete query strings | All eight included | Not included |
| Coordinate preservation | All eight match the reference | Cannot be assessed |
| Required output format | A readable table was returned | JSON was requested but not returned |
| Explanations | Correctly addressed the negative UCB for Function 6, but did not explain every selection individually | Discussed the prompt and validation rather than returning the requested selections |

The simple response selected the highest-UCB candidate for every function and preserved the complete query strings. It also correctly recognised that a negative UCB can still be the highest available score, as in Function 6.

The supplied structured response identified the correct search options, but omitted the query coordinates and did not return valid JSON. It claimed that notebook checks had passed and referred to a complete JSON response, but the supplied text does not demonstrate those claims. I therefore did not treat them as verified results.

This comparison illustrates that explicit instructions do not guarantee compliance. Selection accuracy, completeness and format compliance must be checked separately. A response can identify the correct search options while remaining unsuitable for automated extraction or portal submission.

These observations apply only to the responses reviewed here. One response per prompt is insufficient to establish that either prompting approach is generally more reliable. Conversation context may also influence the responses, so differences cannot be attributed solely to prompt structure.

The final query selection remains based on the maximum UCB calculated in Python. The LLM responses provide an additional review layer; they do not establish the true function values or guarantee improvement.

In [12]:
# Select the highest-UCB candidate for each function.
round_8_selected_proposals = {}
final_rows = []

for function in range(1, 9):
    proposals = round_8_candidate_proposals[function]

    selected_search = max(
        proposals,
        key=lambda name: proposals[name]["ucb"]
    )
    selected = proposals[selected_search]

    query_string = "-".join(
        f"{value:.6f}" for value in selected["query"]
    )

    round_8_selected_proposals[function] = {
        **selected,
        "search": selected_search,
        "query_string": query_string
    }

    final_rows.append({
        "Function": function,
        "Selected search": selected_search,
        "Query for submission": query_string,
        "Predicted mean": selected["mean"],
        "Predicted std": selected["std"],
        "UCB": selected["ucb"]
    })

round_8_final_summary = pd.DataFrame(final_rows).set_index("Function")
round_8_submission_table = round_8_final_summary[
    ["Query for submission"]
].copy()

with pd.option_context("display.max_colwidth", None):
    display(round_8_submission_table)

for function, row in round_8_submission_table.iterrows():
    print(f"Function {function}: {row['Query for submission']}")

,Query for submission
Function,
1,0.208681-0.000255
2,0.698207-0.000240
3,0.991779-0.022828-0.094305
4,0.424455-0.424454-0.381378-0.414254
5,0.997017-0.998912-0.999252-0.996899
6,0.424979-0.403855-0.530055-0.676776-0.000314
7,0.019096-0.226554-0.575958-0.153734-0.332502-0.733585
8,0.149147-0.311840-0.192931-0.145640-0.987373-0.581956-0.118912-0.792680


Function 1: 0.208681-0.000255
Function 2: 0.698207-0.000240
Function 3: 0.991779-0.022828-0.094305
Function 4: 0.424455-0.424454-0.381378-0.414254
Function 5: 0.997017-0.998912-0.999252-0.996899
Function 6: 0.424979-0.403855-0.530055-0.676776-0.000314
Function 7: 0.019096-0.226554-0.575958-0.153734-0.332502-0.733585
Function 8: 0.149147-0.311840-0.192931-0.145640-0.987373-0.581956-0.118912-0.792680
